<a href="https://colab.research.google.com/github/MicheleQGF/ProyectoRappiPlus/blob/main/S12_Estudiante_Proyecto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  


---

## 🔹 Paso 1: Cargar y validar la calidad de los datos



In [ ]:
# importar librerías
import pandas as pd

In [ ]:
# cargar archivos
orders = pd.read_csv("datasets/rappiplus_orders_raw.csv")
catalog = pd.read_csv("datasets/rappiplus_catalog.csv")
marketing = pd.read_csv("datasets/rappiplus_marketing_spend.csv")

In [ ]:
# explorar orders
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [ ]:
orders.info(show_counts=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [ ]:
#columnas numéricas de orders
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [ ]:

#columnas categóricas de orders
columnas_cat_orders = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']


for col in columnas_cat_orders:
    print(f"\n🔍 ANÁLISIS DE: {col.upper()}")
    print("=" * 40)

    # Conteo con valores nulos
    conteo = orders[col].value_counts(dropna=False)
    print(conteo)

    # Porcentaje de valores faltantes
    missing_pct = (orders[col].isnull().sum() / len(orders)) * 100
    print(f"\n⚠️ Valores faltantes: {missing_pct:.1f}%")




🔍 ANÁLISIS DE: PAIS
Colombia     7520
Mexico       7502
Argentina    7291
mexico        865
colombia      823
argentina     799
NaN           300
Name: pais, dtype: int64

⚠️ Valores faltantes: 1.2%

🔍 ANÁLISIS DE: DISPOSITIVO
desktop    12759
mobile     12321
NaN           20
Name: dispositivo, dtype: int64

⚠️ Valores faltantes: 0.1%

🔍 ANÁLISIS DE: FUENTE_REFERENCIA
social         8428
organic        8329
paid_search    8313
NaN              30
Name: fuente_referencia, dtype: int64

⚠️ Valores faltantes: 0.1%

🔍 ANÁLISIS DE: NOMBRE_PRODUCTO
Vacuum-Pro-Black        4199
Blender-XL-Red          4195
Jacket-Winter-M         4192
Sneakers-Urban-42       4160
Laptop-Gaming-16GB      2794
Tablet-Standard-64GB    2780
Phone-Pro-128GB         2750
NaN                       30
Name: nombre_producto, dtype: int64

⚠️ Valores faltantes: 0.1%

🔍 ANÁLISIS DE: CATEGORIA_PRODUCTO
Hogar          8385
Moda           8323
Electronica    8312
NaN              80
Name: categoria_producto, dtype: int64

In [ ]:
print('conteo de id_pedido unicos: ', orders['id_pedido'].nunique())
print('conteo de id_usuario unicos:', orders['id_usuario'].nunique())

conteo de id_pedido unicos:  25000
conteo de id_usuario unicos: 7642


In [ ]:
#Explorar catalog
catalog.head()


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [ ]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [ ]:
#columna numérica de catalog
catalog.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [ ]:
#columnas categóricas de catalog
columnas_cat_catalog = ['nombre_producto', 'categoria_producto','proveedor']

for col in columnas_cat_catalog:
    print(f"\n🔍 ANÁLISIS DE: {col.upper()}")
    print("=" * 40)

    # Conteo con valores nulos
    conteo = catalog[col].value_counts(dropna=False)
    print(conteo)

    # Porcentaje de valores faltantes
    missing_pct = (catalog[col].isnull().sum() / len(catalog)) * 100
    print(f"\n⚠️ Valores faltantes: {missing_pct:.1f}%")


🔍 ANÁLISIS DE: NOMBRE_PRODUCTO
Sneakers-Urban-42       1
Laptop-Gaming-16GB      1
Blender-XL-Red          1
Vacuum-Pro-Black        1
Tablet-Standard-64GB    1
Phone-Pro-128GB         1
Jacket-Winter-M         1
Name: nombre_producto, dtype: int64

⚠️ Valores faltantes: 0.0%

🔍 ANÁLISIS DE: CATEGORIA_PRODUCTO
Electrónica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64

⚠️ Valores faltantes: 0.0%

🔍 ANÁLISIS DE: PROVEEDOR
Bowers LLC                 1
King Ltd                   1
Long-Reid                  1
Mcmillan-Rhodes            1
Fuller, Pena and Myers     1
Rivera, Carr and Finley    1
Greene-Smith               1
Name: proveedor, dtype: int64

⚠️ Valores faltantes: 0.0%


In [ ]:
#Explorar Marketing
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [ ]:
marketing.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [ ]:
#columna numérica marketing
marketing.describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [ ]:
# columnas categóricas marketing
columnas_cat_marketing =['pais', 'canal']
for col in columnas_cat_marketing:
    print(f"\n🔍 ANÁLISIS DE: {col.upper()}")
    print("=" * 40)

    # Conteo con valores nulos
    conteo = marketing[col].value_counts(dropna=False)
    print(conteo)

    # Porcentaje de valores faltantes
    missing_pct = (marketing[col].isnull().sum() / len(marketing)) * 100
    print(f"\n⚠️ Valores faltantes: {missing_pct:.1f}%")


🔍 ANÁLISIS DE: PAIS
Mexico       540
Colombia     540
Argentina    540
Name: pais, dtype: int64

⚠️ Valores faltantes: 0.0%

🔍 ANÁLISIS DE: CANAL
paid_search    507
organic        506
social         506
NaN            101
Name: canal, dtype: int64

⚠️ Valores faltantes: 6.2%


In [ ]:
def resumen_calidad_datos(df, nombre_dataset):
    print(f"\n🔍 ANÁLISIS DE CALIDAD: {nombre_dataset.upper()}")
    print("="*50)

    # Información básica
    print(f"📊 Total registros: {len(df):,}")
    print(f"📊 Total columnas: {len(df.columns)}")

    # Valores faltantes
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"\n⚠️  VALORES FALTANTES:")
        for col, count in missing[missing > 0].items():
            porcentaje = (count/len(df))*100
            print(f"   • {col}: {count:,} ({porcentaje:.1f}%)")
    else:
        print(f"\n✅ Sin valores faltantes")

    # Duplicados
    duplicados = df.duplicated().sum()
    if duplicados > 0:
        print(f"\n⚠️  DUPLICADOS: {duplicados:,}")
    else:
        print(f"\n✅ Sin duplicados")

# Usar la función
# Crear diccionario con nombres y DataFrames
datasets_dict = {
    "Orders": orders,
    "Catalog": catalog,
    "Marketing": marketing
}

# Iterar usando el diccionario
for nombre, df in datasets_dict.items():
    resumen_calidad_datos(df, nombre)




🔍 ANÁLISIS DE CALIDAD: ORDERS
📊 Total registros: 25,100
📊 Total columnas: 12

⚠️  VALORES FALTANTES:
   • pais: 300 (1.2%)
   • dispositivo: 20 (0.1%)
   • fuente_referencia: 30 (0.1%)
   • nombre_producto: 30 (0.1%)
   • categoria_producto: 80 (0.3%)
   • cantidad: 50 (0.2%)
   • precio_unitario: 50 (0.2%)
   • monto_descuento: 50 (0.2%)

⚠️  DUPLICADOS: 100

🔍 ANÁLISIS DE CALIDAD: CATALOG
📊 Total registros: 7
📊 Total columnas: 4

✅ Sin valores faltantes

✅ Sin duplicados

🔍 ANÁLISIS DE CALIDAD: MARKETING
📊 Total registros: 1,620
📊 Total columnas: 5

⚠️  VALORES FALTANTES:
   • canal: 101 (6.2%)

✅ Sin duplicados


---

### Revisión y calidad de datos

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas


In [ ]:


# Validar y convertir fechas al formato correcto
#para orders
orders['fecha_hora_pedido'] =  pd.to_datetime(orders["fecha_hora_pedido"], errors="coerce")

print(f"Tipo de datos de fecha_hora_pedido: {orders['fecha_hora_pedido'].dtype}")
print("Estadísticas de fecha_hora_pedido:")
print(orders['fecha_hora_pedido'].describe())
print(f"Total de fechas únicas: {orders['fecha_hora_pedido'].nunique()}")
# Si hay NaT (Not a Time), significa que algunas fechas no se pudieron convertir
print(f"Fechas inválidas (NaT): {orders['fecha_hora_pedido'].isna().sum()}")





Tipo de datos de fecha_hora_pedido: datetime64[ns]
Estadísticas de fecha_hora_pedido:
count                   25100
unique                    181
top       2025-06-25 00:00:00
freq                      176
first     2025-01-01 00:00:00
last      2025-06-30 00:00:00
Name: fecha_hora_pedido, dtype: object
Total de fechas únicas: 181
Fechas inválidas (NaT): 0


In [ ]:
#para marketing
marketing['fecha'] =  pd.to_datetime(marketing['fecha'], errors="coerce")
print(f"Tipo de datos de fecha: {marketing['fecha'].dtype}")
print("Estadísticas de fecha:")
print(marketing['fecha'].describe())
print(f"Total de fechas únicas: {marketing['fecha'].nunique()}")
# Si hay NaT (Not a Time), significa que algunas fechas no se pudieron convertir
print(f"Fechas inválidas (NaT): {marketing['fecha'].isna().sum()}")


Tipo de datos de fecha: datetime64[ns]
Estadísticas de fecha:
count                    1620
unique                    180
top       2025-06-25 00:00:00
freq                        9
first     2025-01-01 00:00:00
last      2025-06-29 00:00:00
Name: fecha, dtype: object
Total de fechas únicas: 180
Fechas inválidas (NaT): 0


In [ ]:
#Revisar variables numéricas (sin negativos o ceros inválidos)

def analizar_valores_problematicos(df, columnas_numericas, nombre_dataset):
    print(f"\n🔍 ANÁLISIS DE VALORES PROBLEMÁTICOS: {nombre_dataset.upper()}")
    print("=" * 60)

    for col in columnas_numericas:
        if col in df.columns:
            negativos = (df[col] < 0).sum()
            ceros = (df[col] == 0).sum()

            print(f"\n📊 {col}:")
            print(f"   ❌ Negativos: {negativos}")
            print(f"   ⚪ Ceros: {ceros}")
            print(f"   📈 Rango: {df[col].min()} a {df[col].max()}")

            # Si hay valores problemáticos, mostrar ejemplos
            if negativos > 0:
                print(f"   🔍 Ejemplos negativos:")
                print(f"   {df[df[col] < 0][col].head(3).tolist()}")

# Aplicar a todos los datasets
analizar_valores_problematicos(orders, ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total'], "Orders")
analizar_valores_problematicos(catalog, ['costo_unitario'], "Catalog")
analizar_valores_problematicos(marketing, ['gasto'], "Marketing")


🔍 ANÁLISIS DE VALORES PROBLEMÁTICOS: ORDERS

📊 cantidad:
   ❌ Negativos: 4
   ⚪ Ceros: 0
   📈 Rango: -2.0 a 20000.0
   🔍 Ejemplos negativos:
   [-2.0, -1.0, -1.0]

📊 precio_unitario:
   ❌ Negativos: 0
   ⚪ Ceros: 0
   📈 Rango: 20.03 a 499.96

📊 monto_descuento:
   ❌ Negativos: 0
   ⚪ Ceros: 12551
   📈 Rango: 0.0 a 15.0

📊 monto_total:
   ❌ Negativos: 4
   ⚪ Ceros: 0
   📈 Rango: -492.65 a 8840200.0
   🔍 Ejemplos negativos:
   [-192.62, -38.5, -492.65]

🔍 ANÁLISIS DE VALORES PROBLEMÁTICOS: CATALOG

📊 costo_unitario:
   ❌ Negativos: 0
   ⚪ Ceros: 0
   📈 Rango: 10.12 a 280.68

🔍 ANÁLISIS DE VALORES PROBLEMÁTICOS: MARKETING

📊 gasto:
   ❌ Negativos: 0
   ⚪ Ceros: 0
   📈 Rango: 501.11 a 2999.36


In [ ]:
# Filtrar registros con valores negativos en orders
negativos_cantidad = orders[orders['cantidad'] < 0]
negativos_monto = orders[orders['monto_total'] < 0]


print("🔍 REGISTROS CON CANTIDAD NEGATIVA:")
print(negativos_cantidad[['id_pedido', 'nombre_producto', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']])

print("\n🔍 REGISTROS CON MONTO TOTAL NEGATIVO:")
print(negativos_monto[['id_pedido', 'nombre_producto', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']])


🔍 REGISTROS CON CANTIDAD NEGATIVA:
     id_pedido  nombre_producto  cantidad  precio_unitario  monto_descuento  \
266  order_266  Phone-Pro-128GB      -2.0           101.31             10.0   
267  order_267  Phone-Pro-128GB      -1.0            43.50              5.0   
268  order_268  Phone-Pro-128GB      -1.0           497.65              5.0   
269  order_269  Phone-Pro-128GB      -1.0           423.53              0.0   

     monto_total  
266      -192.62  
267       -38.50  
268      -492.65  
269      -423.53  

🔍 REGISTROS CON MONTO TOTAL NEGATIVO:
     id_pedido  nombre_producto  cantidad  precio_unitario  monto_descuento  \
266  order_266  Phone-Pro-128GB      -2.0           101.31             10.0   
267  order_267  Phone-Pro-128GB      -1.0            43.50              5.0   
268  order_268  Phone-Pro-128GB      -1.0           497.65              5.0   
269  order_269  Phone-Pro-128GB      -1.0           423.53              0.0   

     monto_total  
266      -192.62  
2

In [ ]:
#corrección de negativos en orders

orders['cantidad'] = orders['cantidad'].abs()
orders['monto_total'] = orders['monto_total'].abs()

# Verificar que ya no hay negativos
print("Verificación después de la corrección:")
print(f"Cantidad negativa: {(orders['cantidad'] < 0).sum()}")
print(f"Monto total negativo: {(orders['monto_total'] < 0).sum()}")

# Ver los registros que se corrigieron
print("\nRegistros corregidos:")
registros_corregidos = orders.loc[[266, 267, 268, 269]]
print(registros_corregidos[['id_pedido', 'cantidad', 'monto_total']])


Verificación después de la corrección:
Cantidad negativa: 0
Monto total negativo: 0

Registros corregidos:
     id_pedido  cantidad  monto_total
266  order_266       2.0       192.62
267  order_267       1.0        38.50
268  order_268       1.0       492.65
269  order_269       1.0       423.53


In [ ]:

#Verificar consistencia de montos
# El monto total debería ser: (precio_unitario × cantidad) - monto_descuento
orders['monto_calculado'] = (orders['precio_unitario'] * orders['cantidad']) - orders['monto_descuento']
orders['diferencia'] = orders['monto_total'] - orders['monto_calculado']

# Verificar si hay diferencias significativas
inconsistencias = orders[abs(orders['diferencia']) > 0.01]  # Tolerancia de 1 centavo
print(f"Registros con inconsistencias: {len(inconsistencias)}")


Registros con inconsistencias: 1146


In [ ]:
# Qué tan grandes son las diferencias
print("📊 ANÁLISIS DE LAS 1146 INCONSISTENCIAS:")
print("=" * 50)
print(f"Diferencia mínima: {inconsistencias['diferencia'].min():.2f}")
print(f"Diferencia máxima: {inconsistencias['diferencia'].max():.2f}")
print(f"Diferencia promedio: {inconsistencias['diferencia'].mean():.2f}")
print(f"Diferencia mediana: {inconsistencias['diferencia'].median():.2f}")

# Distribución de las diferencias
print("\n📈 DISTRIBUCIÓN DE DIFERENCIAS:")
print(inconsistencias['diferencia'].describe())

📊 ANÁLISIS DE LAS 1146 INCONSISTENCIAS:
Diferencia mínima: -0.01
Diferencia máxima: 0.01
Diferencia promedio: 0.00
Diferencia mediana: 0.01

📈 DISTRIBUCIÓN DE DIFERENCIAS:
count    1146.000000
mean        0.000436
std         0.009995
min        -0.010000
25%        -0.010000
50%         0.010000
75%         0.010000
max         0.010000
Name: diferencia, dtype: float64


In [ ]:
# Ver ejemplos de registros problemáticos
print("\n🔍 EJEMPLOS DE REGISTROS CON INCONSISTENCIAS:")
ejemplos = inconsistencias[['id_pedido', 'cantidad', 'precio_unitario', 'monto_descuento',
                          'monto_total', 'monto_calculado', 'diferencia']].head(10)
print(ejemplos)


🔍 EJEMPLOS DE REGISTROS CON INCONSISTENCIAS:
     id_pedido  cantidad  precio_unitario  monto_descuento  monto_total  \
2      order_2       2.0           102.99             10.0       195.99   
24    order_24       2.0           117.99              0.0       235.99   
35    order_35       2.0           467.34              5.0       929.69   
41    order_41       2.0            37.77             10.0        65.53   
136  order_136       2.0           211.05             15.0       407.09   
180  order_180       2.0           451.16             15.0       887.31   
261  order_261       2.0           209.33             10.0       408.65   
270  order_270       2.0           127.95              0.0       255.89   
281  order_281       2.0           100.68              0.0       201.35   
290  order_290       2.0            41.92              5.0        78.83   

     monto_calculado  diferencia  
2             195.98        0.01  
24            235.98        0.01  
35            929.68   

In [ ]:
# Solo corrige los que tienen diferencias > 0.01
mask_inconsistentes = abs(orders['diferencia']) > 0.01
orders.loc[mask_inconsistentes, 'monto_total'] = (
    orders.loc[mask_inconsistentes, 'precio_unitario'] *
    orders.loc[mask_inconsistentes, 'cantidad'] -
    orders.loc[mask_inconsistentes, 'monto_descuento']
)

In [ ]:
#Verificar consistencia de montos
# El monto total debería ser: (precio_unitario × cantidad) - monto_descuento
orders['monto_calculado'] = (orders['precio_unitario'] * orders['cantidad']) - orders['monto_descuento']
orders['diferencia'] = orders['monto_total'] - orders['monto_calculado']

# Verificar si hay diferencias significativas
inconsistencias = orders[abs(orders['diferencia']) > 0.01]  # Tolerancia de 1 centavo
print(f"Registros con inconsistencias: {len(inconsistencias)}")


Registros con inconsistencias: 0


In [ ]:
#Eliminar duplicados en orders
orders = orders.drop_duplicates()
print('duplicados:',df.duplicated().sum())
orders = orders.drop(['diferencia', 'monto_calculado'], axis=1)
orders.info()


duplicados: 0
<class 'pandas.core.frame.DataFrame'>
Int64Index: 25000 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25000 non-null  object        
 1   id_usuario          25000 non-null  object        
 2   fecha_hora_pedido   25000 non-null  datetime64[ns]
 3   pais                24700 non-null  object        
 4   dispositivo         24980 non-null  object        
 5   fuente_referencia   24970 non-null  object        
 6   nombre_producto     24970 non-null  object        
 7   categoria_producto  24920 non-null  object        
 8   cantidad            24950 non-null  float64       
 9   precio_unitario     24950 non-null  float64       
 10  monto_descuento     24950 non-null  float64       
 11  monto_total         25000 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


In [ ]:

# Estandarizar países en orders
orders['pais'] = orders['pais'].str.title()

# Verificar el resultado
print("Países después de aplicar estandarización:")
print(orders['pais'].value_counts(dropna=False))


Países después de aplicar estandarización:
Mexico       8341
Colombia     8304
Argentina    8055
NaN           300
Name: pais, dtype: int64


In [ ]:
# Revisión de nulos en columnas categóricas de orders
# Revisar si los faltantes se concentran en ciertos grupos (MAR)

# 1. Crear un resumen de valores faltantes por columna
print("🔍 RESUMEN DE VALORES FALTANTES")
print("=" * 50)

missing_summary = orders.isnull().sum()
missing_pct = (orders.isnull().sum() / len(orders)) * 100

# Crear DataFrame con el resumen
resumen_faltantes = pd.DataFrame({
    'columna': missing_summary.index,
    'valores_faltantes': missing_summary.values,
    'porcentaje': missing_pct.values
})

# Filtrar solo las que tienen faltantes
resumen_faltantes = resumen_faltantes[resumen_faltantes['valores_faltantes'] > 0]
resumen_faltantes = resumen_faltantes.sort_values('valores_faltantes', ascending=False)

print(resumen_faltantes)



🔍 RESUMEN DE VALORES FALTANTES
               columna  valores_faltantes  porcentaje
3                 pais                300        1.20
7   categoria_producto                 80        0.32
8             cantidad                 50        0.20
9      precio_unitario                 50        0.20
10     monto_descuento                 50        0.20
5    fuente_referencia                 30        0.12
6      nombre_producto                 30        0.12
4          dispositivo                 20        0.08


Los porcentajes son bajos, por lo que decidimos eliminar esos registros


In [ ]:
# Mantener solo registros completos y convertimos cantidades a int

orders = orders.dropna()
orders['cantidad'] = orders['cantidad'].astype(int)
orders.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 24600 entries, 0 to 24999
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24600 non-null  object        
 1   id_usuario          24600 non-null  object        
 2   fecha_hora_pedido   24600 non-null  datetime64[ns]
 3   pais                24600 non-null  object        
 4   dispositivo         24600 non-null  object        
 5   fuente_referencia   24600 non-null  object        
 6   nombre_producto     24600 non-null  object        
 7   categoria_producto  24600 non-null  object        
 8   cantidad            24600 non-null  int64         
 9   precio_unitario     24600 non-null  float64       
 10  monto_descuento     24600 non-null  float64       
 11  monto_total         24600 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(7)
memory usage: 2.4+ MB


In [ ]:

# Completar registros de categoria de producto con nombre_producto de catalog
categorias_ref = catalog.set_index('nombre_producto')['categoria_producto'].to_dict()
# Imputar solo donde precio_unitario es nulo
orders.loc[
    orders['categoria_producto'].isna(),
    'categoria_producto'
] = orders.loc[
    orders['categoria_producto'].isna(),
    'nombre_producto'
].map(categorias_ref)
print('categoria_producto con nulos restantes: ',orders['categoria_producto'].isna().sum())

#Homologar acento en Electrónica
# Crear diccionario de mapeo para estandarizar
mapeo_categorias = {
    'Electronica': 'Electrónica',
    'electronica': 'Electrónica',
    'electrónica': 'Electrónica'
}

# Aplicar el mapeo
orders['categoria_producto'] = orders['categoria_producto'].replace(mapeo_categorias)

print("Categorías después del mapeo:")
print(orders['categoria_producto'].value_counts())


categoria_producto con nulos restantes:  0
Categorías después del mapeo:
Hogar          8228
Moda           8194
Electrónica    8178
Name: categoria_producto, dtype: int64


In [ ]:
# 1. Crear nuevo resumen de valores faltantes por columna
print("🔍 NUEVO RESUMEN DE VALORES FALTANTES")
print("=" * 50)

missing_summary = orders.isnull().sum()
missing_pct = (orders.isnull().sum() / len(orders)) * 100

# Crear DataFrame con el resumen
resumen_faltantes = pd.DataFrame({
    'columna': missing_summary.index,
    'valores_faltantes': missing_summary.values,
    'porcentaje': missing_pct.values
})

# Filtrar solo las que tienen faltantes
resumen_faltantes = resumen_faltantes[resumen_faltantes['valores_faltantes'] > 0]
resumen_faltantes = resumen_faltantes.sort_values('valores_faltantes', ascending=False)

print(resumen_faltantes)

🔍 NUEVO RESUMEN DE VALORES FALTANTES
Empty DataFrame
Columns: [columna, valores_faltantes, porcentaje]
Index: []


Quitamos los registros faltantes en `orders` porque los porcentajes son muy bajos

In [ ]:
# Verificar si hay variación en precios unitarios por producto
print("🔍 VERIFICACIÓN DE CONSISTENCIA DE PRECIOS UNITARIOS:")
print("=" * 50)

variacion_precios = orders.groupby('nombre_producto')['precio_unitario'].agg(['min', 'max', 'std'])
variacion_precios['diferencia'] = variacion_precios['max'] - variacion_precios['min']

# Mostrar productos con variación en precios
productos_con_variacion = variacion_precios[variacion_precios['diferencia'] > 0.01]

if len(productos_con_variacion) > 0:
    print("⚠️ Productos con precios variables:")
    print(productos_con_variacion)
else:
    print("✅ Todos los productos tienen precios consistentes")

🔍 VERIFICACIÓN DE CONSISTENCIA DE PRECIOS UNITARIOS:
⚠️ Productos con precios variables:
                        min     max         std  diferencia
nombre_producto                                            
Blender-XL-Red        20.06  499.94  137.535203      479.88
Jacket-Winter-M       20.06  499.93  138.937186      479.87
Laptop-Gaming-16GB    20.24  499.95  140.048032      479.71
Phone-Pro-128GB       20.08  499.86  138.063140      479.78
Sneakers-Urban-42     20.11  499.91  138.525756      479.80
Tablet-Standard-64GB  20.03  499.89  138.638905      479.86
Vacuum-Pro-Black      20.37  499.96  139.309116      479.59


Los precios unitarios son variables y no tenemos datos de precio unitario en catalog por lo que utilizaremos los montos totales para hacer los cálculos del análisis, sobre todo los márgenes sobre costo por producto


In [ ]:
#Para marketing llenar con n/a los registros nulos de canal
marketing['canal'].fillna('N/A', inplace=True)
print('Canal con nulos restantes: ', marketing['canal'].isna().sum())
marketing.info()

Canal con nulos restantes:  0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


Conservamos los nulos de `canal`en `marketing` porque 6.2% ya puede ser significativo

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:


# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)



---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# Ingreso Total en los primeros 6 meses de 2025
ingreso_total = orders['monto_total'].sum()
print(f"💰 Ingreso total de enero a junio del 2025: ${ingreso_total:,.2f}")


💰 Ingreso total de enero a junio del 2025: $51,836,374.55


In [ ]:
#Ingreso por País en los primeros 6 meses del año
ingreso_pais= orders.groupby('pais')['monto_total'].sum().sort_values(ascending=False)
print("🏆 Ingreso por País:")
print(ingreso_pais.head())


🏆 Ingreso por País:
pais
Argentina    20719056.73
Mexico       19751948.48
Colombia     11365369.34
Name: monto_total, dtype: float64


In [ ]:
#union de orders con catalog para obtener los costos unitarios en orders
orders_merged = pd.merge(orders, catalog, on=['nombre_producto'], how='inner')
# Mantener solo la columna de orders y eliminar la de catalog
orders_merged = orders_merged.drop('categoria_producto_y', axis=1)
orders_merged = orders_merged.rename(columns={'categoria_producto_x': 'categoria_producto'})
# Crear columna de costo total
orders_merged['costo_total'] = orders_merged['cantidad'] * orders_merged['costo_unitario']
orders_merged.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,proveedor,costo_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2,332.69,0.0,665.37,189.31,Mcmillan-Rhodes,378.62
1,order_12,user_6810,2025-03-17,Colombia,mobile,social,Jacket-Winter-M,Moda,1,499.48,5.0,494.48,189.31,Mcmillan-Rhodes,189.31
2,order_13,user_3170,2025-04-10,Colombia,desktop,organic,Jacket-Winter-M,Moda,1,496.94,0.0,496.94,189.31,Mcmillan-Rhodes,189.31
3,order_15,user_1500,2025-04-03,Colombia,mobile,organic,Jacket-Winter-M,Moda,2,373.47,0.0,746.94,189.31,Mcmillan-Rhodes,378.62
4,order_19,user_7190,2025-02-03,Argentina,mobile,social,Jacket-Winter-M,Moda,2,45.86,0.0,91.72,189.31,Mcmillan-Rhodes,378.62


In [ ]:
#costo Total en los primeros 6 meses de 2025
costo_total = orders_merged['costo_total'].sum()
print(f"💰 Costo total de enero a junio del 2025: ${costo_total:,.2f}")

💰 Costo total de enero a junio del 2025: $43,078,678.80


In [ ]:
# gasto total en marketing en los primeros 6 meses de 2025
gasto_total_marketing = marketing['gasto'].sum()
print(f"💰 Gasto en Marketing total de enero a junio del 2025: ${gasto_total_marketing:,.2f}")

💰 Gasto en Marketing total de enero a junio del 2025: $2,871,843.53


In [ ]:

#gasto en Marketing por País en los primeros 6 meses del año
gasto_marketing_pais= marketing.groupby('pais')['gasto'].sum().sort_values(ascending=False).reset_index()
print("🏆 Gasto Marketing por País:")
print(gasto_marketing_pais.head())


🏆 Gasto Marketing por País:
        pais      gasto
0     Mexico  988495.51
1  Argentina  947694.60
2   Colombia  935653.42


In [ ]:
# utilidad en los primeros 6 meses del 2025
utilidad = ingreso_total - costo_total - gasto_total_marketing
print(f"💰 Utilidad de enero a junio del 2025: ${utilidad:,.2f}")

💰 Utilidad de enero a junio del 2025: $5,885,852.22


In [ ]:
# Margen en los primeros 6 meses del 2025
margen = utilidad/ingreso_total*100
print(f"📈 Margen de enero a junio del 2025: {margen:,.2f}%")

📈 Margen de enero a junio del 2025: 11.35%


RappiPlus tuvo una rentabilidad sobre las ventas del 11.33% de enero a junio del 2025

In [ ]:
#Utilidad y Margen por País en los primeros 6 meses del 2025
# Agrupar orders_merged por país para obtener ingresos y costos
utilidad_por_pais = orders_merged.groupby('pais').agg({
    'monto_total': 'sum',      # Ingresos totales
    'costo_total': 'sum'       # Costos totales
}).reset_index()

# Renombrar columnas para mayor claridad
utilidad_por_pais.columns = ['pais', 'ingresos', 'costos']

# Utilizamos el gasto calculado anteriormente
gasto_marketing_pais.columns = ['pais', 'gasto_marketing']

# Unir con la tabla de utilidad
utilidad_por_pais = pd.merge(utilidad_por_pais, gasto_marketing_pais, on='pais', how='left')

# Calcular utilidad por país
utilidad_por_pais['utilidad'] = (
    utilidad_por_pais['ingresos'] -
    utilidad_por_pais['costos'] -
    utilidad_por_pais['gasto_marketing']
)

# Calcular margen por país (%)
utilidad_por_pais['margen_pct'] = (
    utilidad_por_pais['utilidad'] / utilidad_por_pais['ingresos'] * 100
).round(2)

# Mostrar resultados
print("💰 UTILIDAD Y MARGEN POR PAÍS:")
print("=" * 50)
print(utilidad_por_pais)

💰 UTILIDAD Y MARGEN POR PAÍS:
        pais     ingresos        costos  gasto_marketing      utilidad  \
0  Argentina  20719056.73  1.526997e+07        947694.60  4.501393e+06   
1   Colombia  11365369.34  9.690263e+06        935653.42  7.394528e+05   
2     Mexico  19751948.48  1.811845e+07        988495.51  6.450065e+05   

   margen_pct  
0       21.73  
1        6.51  
2        3.27  


In [ ]:
#Ticket Promedio por orden
ticket_promedio = orders['monto_total'].mean()
print(f"🎫 Ticket promedio: ${ticket_promedio:.2f}")

🎫 Ticket promedio: $2107.17


In [ ]:
#cantidad promedio de productos por orden
cantidad_promedio = orders['cantidad'].mean()
print(f"📦 Cantidad promedio por orden: {cantidad_promedio:.1f}")

📦 Cantidad promedio por orden: 7.2


In [ ]:
#Producto más vendido por cantidad
producto_mas_vendido_cantidad = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
print("🏆 Producto más vendido por CANTIDAD:")
print(producto_mas_vendido_cantidad.head())

🏆 Producto más vendido por CANTIDAD:
nombre_producto
Laptop-Gaming-16GB    144160
Vacuum-Pro-Black        6203
Jacket-Winter-M         6185
Blender-XL-Red          6184
Sneakers-Urban-42       6085
Name: cantidad, dtype: int64


In [ ]:

#producto más vendido por número de órdenes
producto_mas_vendido_ordenes = orders['nombre_producto'].value_counts()
print("🏆 Producto más vendido por NÚMERO DE ÓRDENES:")
print(producto_mas_vendido_ordenes.head())


🏆 Producto más vendido por NÚMERO DE ÓRDENES:
Jacket-Winter-M       4122
Vacuum-Pro-Black      4114
Blender-XL-Red        4114
Sneakers-Urban-42     4072
Laptop-Gaming-16GB    2752
Name: nombre_producto, dtype: int64


In [ ]:
#Producto que genera más ingresos

producto_mas_vendido_revenue = orders.groupby('nombre_producto')['monto_total'].sum().sort_values(ascending=False)
print("🏆 Producto que genera más ingresos:")
print(producto_mas_vendido_revenue.head())


🏆 Producto que genera más ingresos:
nombre_producto
Laptop-Gaming-16GB    43420726.58
Vacuum-Pro-Black       1602954.77
Jacket-Winter-M        1588706.29
Blender-XL-Red         1587286.62
Sneakers-Urban-42      1542350.47
Name: monto_total, dtype: float64


In [ ]:
#Margen sobre costo por Producto

margen_por_producto = (orders_merged.groupby('nombre_producto')['monto_total'].sum()-orders_merged.groupby('nombre_producto')['costo_total'].sum())/orders_merged.groupby('nombre_producto')['monto_total'].sum()*100


print("📊 MARGEN SOBRE COSTO POR PRODUCTO:")
print(margen_por_producto)

📊 MARGEN SOBRE COSTO POR PRODUCTO:
nombre_producto
Blender-XL-Red          31.181820
Jacket-Winter-M         26.299634
Laptop-Gaming-16GB       6.812179
Phone-Pro-128GB         96.037114
Sneakers-Urban-42       93.210178
Tablet-Standard-64GB    90.140639
Vacuum-Pro-Black        93.576250
dtype: float64


Aunque el producto con más cantidades vendidas y el que más ingresos generó es `Laptop_gaming_16GB`, su margen es sólo del 7% contra el `Phone-pro-128GB`, o la `Vacuum_Pro_Black`, o los `Sneakers-Urban-42` que tienen un margen del 96%, 94% y 93% respectivamente. Además podríamos estar frente a pedidos muy altos de una sola vez y no sostenibles a largo plazo para la `Laptop_gaming_16GB`, auqnue no tenemos suficientes datos históricos como para verificarlo

In [ ]:
#Cuánto se ha gastado en Marketing por canal
gasto_marketing_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)
print("🏆 Gasto en Marketing por canal:")
print(gasto_marketing_canal.head())


🏆 Gasto en Marketing por canal:
canal
social         918043.21
organic        913533.01
paid_search    863088.21
N/A            177179.10
Name: gasto, dtype: float64


Recomendamos impulsar productos como el Phone-Pro-128GB (96% de margen) y la Vacuum-Pro-Black (94% de margen). Actualmente tienen un volumen de órdenes saludable pero un impacto mínimo en el ingreso total debido a su bajo precio o menor rotación comparada con las laptops.


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) as total_usuarios
FROM events
GROUP BY nombre_evento
ORDER BY total_usuarios DESC

'''


totals = pd.read_sql(query_totals, con=engine)
totals


,nombre_evento,total_usuarios
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


La discrepancia entre `add_to_cart` y `select_item` podría ser atribuible preliminarmente a recompras o adiciones desde wishlists. También podrían ser errores de registro. Esto son sólo hipótesis.


In [ ]:
# PARTE 2: Conversiones
# ======================



query_conversion = '''

WITH first_visit AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'first_visit'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
select_item AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento ='select_item'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
add_to_cart AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_to_cart'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
begin_checkout AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'begin_checkout'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
add_payment_info AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'add_payment_info'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
purchase AS (
  SELECT DISTINCT id_usuario
  FROM events
  WHERE nombre_evento = 'purchase'
    AND timestamp_evento BETWEEN '2025-01-01' AND '2025-06-30'
),
funnel_counts AS(
SELECT
  COUNT(fv.id_usuario) AS usuarios_first_visit,
  COUNT(si.id_usuario) AS usuarios_select_item,
  COUNT(a.id_usuario) AS usuarios_add_to_cart,
  COUNT(bc.id_usuario) AS usuarios_begin_checkout,
  COUNT(api.id_usuario) AS usuarios_add_payment_info,
  COUNT(p.id_usuario) AS usuarios_purchase
FROM first_visit fv
LEFT JOIN select_item si        ON fv.id_usuario = si.id_usuario
LEFT JOIN add_to_cart a         ON fv.id_usuario = a.id_usuario
LEFT JOIN begin_checkout bc     ON fv.id_usuario = bc.id_usuario
LEFT JOIN add_payment_info api  ON fv.id_usuario = api.id_usuario
LEFT JOIN purchase p            ON fv.id_usuario = p.id_usuario
)


SELECT
    ROUND( usuarios_select_item *100.0 / NULLIF(usuarios_first_visit, 0),2 ) AS conversion_select_item,
    ROUND( usuarios_add_to_cart *100.0 / NULLIF(usuarios_first_visit, 0),2 ) AS conversion_add_to_cart,
    ROUND( usuarios_begin_checkout *100.0 / NULLIF(usuarios_first_visit, 0),2 ) AS conversion_begin_checkout,
    ROUND( usuarios_add_payment_info *100.0 / NULLIF(usuarios_first_visit, 0),2 ) AS conversion_add_payment_info,
    ROUND( usuarios_purchase *100.0 / NULLIF(usuarios_first_visit, 0),2 ) AS conversion_purchase,
    ((SELECT COUNT(*) FROM first_visit) - (SELECT COUNT(*) FROM select_item)) * 100
    / NULLIF((SELECT COUNT(*) FROM first_visit), 0) AS dropoff_after_first_visit_pct,
    ((SELECT COUNT(*) FROM select_item) - (SELECT COUNT(*) FROM add_to_cart)) * 100
    / NULLIF((SELECT COUNT(*) FROM select_item), 0) AS dropoff_after_select_item_pct,
    ((SELECT COUNT(*) FROM add_to_cart) - (SELECT COUNT(*) FROM begin_checkout)) * 100
    / NULLIF((SELECT COUNT(*) FROM add_to_cart), 0) AS dropoff_after_add_to_cart_pct,
    ((SELECT COUNT(*) FROM begin_checkout) - (SELECT COUNT(*) FROM add_payment_info)) * 100
    / NULLIF((SELECT COUNT(*) FROM begin_checkout), 0) AS dropoff_after_begin_checkout_pct,
    ((SELECT COUNT(*) FROM add_payment_info) - (SELECT COUNT(*) FROM purchase)) * 100
    / NULLIF((SELECT COUNT(*) FROM add_payment_info), 0) AS dropoff_after_add_payment_info_pct

FROM funnel_counts

'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion


,conversion_select_item,conversion_add_to_cart,conversion_begin_checkout,conversion_add_payment_info,conversion_purchase,dropoff_after_first_visit_pct,dropoff_after_select_item_pct,dropoff_after_add_to_cart_pct,dropoff_after_begin_checkout_pct,dropoff_after_add_payment_info_pct
0,94.83,95.42,90.19,78.09,77.83,2,0,5,13,0


La conversión del paso de `select_item` a `add_to_cart` es casi perfecta, más de 90% hasta `begin_checkout`. Esto indica que la intención de compra es altísima una vez que el usuario interactúa con el producto.

El mayor punto de abandono ocurre en después de `begin_checkout` con un 13%.

Esto significa que los usuarios inician el proceso de pago, pero se detienen al llegar a la pantalla de información de pago.

Una vez que el usuario supera la barrera del pago (add_payment_info), la conversión a purchase es casi garantizada. El problema no es la decisión de compra, sino el formulario o método de pago.

Se recomienda revisar Métodos de Pago, el dropoff podría deberse hipotéticamente a falta de opciones de pago local, rechazos de tarjetas de débito/crédito o falta de opciones de facilidades de pago. También se sugiere simplificar el Formulario para reducir el esfuerzo en la etapa de `add_payment_info`.


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)


**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
#revisar fecha de users
query_fecha_users = '''
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'users'
AND column_name = 'fecha_registro';
'''

fecha_users = pd.read_sql(query_fecha_users, con=engine)
print(fecha_users)

#fecha_users.head(3)


      column_name data_type
0  fecha_registro      text


In [ ]:
# Explorar tabla users
# =========================
query_users = '''

SELECT *
FROM users;

'''

users = pd.read_sql(query_users, con=engine)
users.head(3)


,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
#revisar fecha de user_activity
query_fecha_user_activity ='''
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'user_activity'
AND column_name = 'fecha_actividad';
'''

fecha_user_activity = pd.read_sql(query_fecha_user_activity, con=engine)
print(fecha_user_activity)


       column_name data_type
0  fecha_actividad      text


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT*
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
query_dias_despues_registro= '''
SELECT DISTINCT dias_despues_registro
FROM user_activity;
 '''
dias_registro= pd.read_sql(query_dias_despues_registro, con=engine)
dias_registro.head()

,dias_despues_registro
0,21
1,7
2,14
3,28


In [ ]:
# Retención por cohortes
# ======================
from sqlalchemy import text

query_cohort_retention_final = '''

WITH cohort AS (
SELECT
    u.id_usuario,
    TO_CHAR(DATE_TRUNC('month', MIN( CAST(fecha_registro AS DATE))), 'YYYY-MM') AS cohort
FROM users u
WHERE CAST(u.fecha_registro AS DATE) BETWEEN '2025-01-01' AND '2025-06-30'
GROUP BY u.id_usuario
),

activity AS (
SELECT
    a.id_usuario,
    c.cohort,
    a.dias_despues_registro,
    a.activo
FROM user_activity AS a
    LEFT JOIN cohort AS c ON a.id_usuario = c.id_usuario
WHERE a.fecha_actividad BETWEEN '2025-01-01'AND '2025-06-30'

),

retencion_base AS (
    SELECT
        cohort,
        COUNT(DISTINCT id_usuario) as total_usuarios,
        COUNT(DISTINCT CASE WHEN dias_despues_registro = 7  AND activo = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN dias_despues_registro = 14 AND activo = 1 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN dias_despues_registro = 21 AND activo = 1 THEN id_usuario END) AS retenido_w3,
        COUNT(DISTINCT CASE WHEN dias_despues_registro = 28 AND activo = 1 THEN id_usuario END) AS retenido_w4
    FROM activity
    GROUP BY cohort
)


SELECT
    cohort,
    ROUND(retenido_w1 * 100.0 / total_usuarios, 1) AS semana_1,
    ROUND(retenido_w2 * 100.0 / total_usuarios, 1) AS semana_2,
    ROUND(retenido_w3 * 100.0 / total_usuarios, 1) AS semana_3,
    ROUND(retenido_w4 * 100.0 / total_usuarios, 1) AS semana_4,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    retenido_w4,
    total_usuarios

FROM retencion_base
ORDER BY cohort;

'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(text(query_cohort_retention_final), con=engine)
cohorte_final


,cohort,semana_1,semana_2,semana_3,semana_4,retenido_w1,retenido_w2,retenido_w3,retenido_w4,total_usuarios
0,2025-01,42.8,41.1,40.3,41.2,697,668,656,671,1627
1,2025-02,42.3,42.2,44.0,39.8,611,609,635,575,1444
2,2025-03,41.4,43.1,42.2,41.1,677,705,690,673,1636
3,2025-04,42.3,43.4,41.3,40.6,680,697,663,652,1606
4,2025-05,41.2,40.1,41.8,40.2,695,676,706,679,1687


Los usuarios que superan la primera semana se quedan en RappiPlus todo el mes, aunque la retención está "estancada" en el rango del 40% al 43%. Esto sugiere que hay un grupo de usuarios muy fiel, pero no se está logrando que los usuarios nuevos aumenten su compromiso con el tiempo (no hay crecimiento incremental en la retención).
Se recomiendan campañas de Re-engagement en las semanas subsecuentes al registro, como la retención es plana, el foco debe estar en subir el porcentaje desde la semana 1.
Se sugiere también implementar recompensas que aumenten según las semanas de uso consecutivo para incentivar el aumento de porcentaje de retención.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No hay diferencia significativa en las tasas de conversión entre la variante de Control y la variante de Tratamiento.
   - **H₁ (Hipótesis alternativa):** Sí hay diferencia significativa en las tasas de conversión entre ambas variantes.
   
**Test estadístico:** Chi-cuadrado de independencia

**Nivel de significancia alpha:** 0.05 (5%)

In [ ]:
#cargar librerías
from scipy import stats
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind
from scipy.stats import pointbiserialr
import numpy as np


In [ ]:
# Cargar dataset
e_checkout = pd.read_csv("datasets/experiment_checkout_ui.csv")


In [ ]:
#Explorar e_checkout
e_checkout.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [ ]:
e_checkout.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


In [ ]:
#estadísticas de columnas numéricas de e_checkout
e_checkout.describe()

,convirtio,duracion_sesion
count,10000.000000,10000.000000
mean,0.159900,159.862439
std,0.366532,81.074410
min,0.000000,20.010000
25%,0.000000,89.470000
50%,0.000000,159.725000
75%,0.000000,229.745000
max,1.000000,300.000000


In [ ]:
# Validar y convertir fechas al formato correcto
#para e_checkout
e_checkout['timestamp'] =  pd.to_datetime(e_checkout["timestamp"], errors="coerce")

print(f"Tipo de datos de timestamp: {e_checkout['timestamp'].dtype}")
print("Estadísticas de timestamp:")
print(e_checkout['timestamp'].describe())
print(f"Total de fechas únicas: {e_checkout['timestamp'].nunique()}")
# Si hay NaT (Not a Time), significa que algunas fechas no se pudieron convertir
print(f"Fechas inválidas (NaT): {e_checkout['timestamp'].isna().sum()}")


Tipo de datos de timestamp: datetime64[ns]
Estadísticas de timestamp:
count                   10000
unique                    181
top       2025-06-06 00:00:00
freq                       75
first     2025-01-01 00:00:00
last      2025-06-30 00:00:00
Name: timestamp, dtype: object
Total de fechas únicas: 181
Fechas inválidas (NaT): 0


In [ ]:
resumen_calidad_datos(e_checkout, 'Experiment Checkout')


🔍 ANÁLISIS DE CALIDAD: EXPERIMENT CHECKOUT
📊 Total registros: 10,000
📊 Total columnas: 7

✅ Sin valores faltantes

✅ Sin duplicados


In [ ]:
#Conteo de usuarios
e_checkout['id_usuario'].nunique()

10000

In [ ]:
# Resumen estadístico de usuarios que se convirtieron
e_checkout['convirtio'].value_counts(normalize=True)

0    0.8401
1    0.1599
Name: convirtio, dtype: float64

In [ ]:
# Explorar variables categóricas y cómo se distribuyen
cols_categoricas=['variante', 'dispositivo','pais']
for col in cols_categoricas:
    print(f"\n{col}:")
    print(e_checkout[col].value_counts(normalize=True))


variante:
tratamiento    0.5035
control        0.4965
Name: variante, dtype: float64

dispositivo:
desktop    0.5042
mobile     0.4958
Name: dispositivo, dtype: float64

pais:
Mexico       0.3405
Argentina    0.3317
Colombia     0.3278
Name: pais, dtype: float64


In [ ]:
# Test estadístico para ver convertidos


# Tabla de contingencia
tabla_contingencia = pd.crosstab(e_checkout['variante'], e_checkout['convirtio'], margins=True)

print("📊 TABLA DE CONTINGENCIA:")
print(tabla_contingencia)



# Tasas de conversión por variante
conversion_rates = e_checkout.groupby('variante')['convirtio'].agg(['count', 'sum', 'mean'])
conversion_rates.columns = ['total_usuarios', 'conversiones', 'tasa_conversion']
conversion_rates['tasa_conversion_pct'] = conversion_rates['tasa_conversion'] * 100
print("\n📈 TASAS DE CONVERSIÓN POR VARIANTE:")
print(conversion_rates)




📊 TABLA DE CONTINGENCIA:
convirtio       0     1    All
variante                      
control      4186   779   4965
tratamiento  4215   820   5035
All          8401  1599  10000

📈 TASAS DE CONVERSIÓN POR VARIANTE:
             total_usuarios  conversiones  tasa_conversion  \
variante                                                     
control                4965           779         0.156898   
tratamiento            5035           820         0.162860   

             tasa_conversion_pct  
variante                          
control                15.689829  
tratamiento            16.285998  


In [ ]:
# Test Chi-cuadrado para independencia
chi2, p_value, dof, expected = chi2_contingency(tabla_contingencia)

print(f"\n🧪 TEST CHI-CUADRADO:")
print(f"Chi-cuadrado: {chi2}")
print(f"P-value: {p_value}")

# Interpretación
alpha = 0.05
if p_value < alpha:
    print(f"✅ Resultado: SIGNIFICATIVO (p < {alpha})")
    print("Hay evidencia de que la variante afecta la conversión")
else:
    print(f"❌ Resultado: NO SIGNIFICATIVO (p >= {alpha})")
    print("No hay evidencia suficiente de que la variante afecte la conversión")


🧪 TEST CHI-CUADRADO:
Chi-cuadrado: 0.661421591043577
P-value: 0.9559998112927427
❌ Resultado: NO SIGNIFICATIVO (p >= 0.05)
No hay evidencia suficiente de que la variante afecte la conversión


In [ ]:

# Análisis de conversión por país
for pais in e_checkout['pais'].unique():
    print(f"\n🌍 ANÁLISIS PARA {pais.upper()}:")
    print("=" * 40)

    # Filtrar datos por país
    data_pais = e_checkout[e_checkout['pais'] == pais]

    # Tabla de contingencia para este país
    tabla_pais = pd.crosstab(data_pais['variante'], data_pais['convirtio'])
    print("Tabla de contingencia:")
    print(tabla_pais)

    # Tasas de conversión por variante en este país
    conversion_pais = data_pais.groupby('variante')['convirtio'].agg(['count', 'sum', 'mean'])
    conversion_pais.columns = ['total_usuarios', 'conversiones', 'tasa_conversion']
    conversion_pais['tasa_conversion_pct'] = conversion_pais['tasa_conversion'] * 100
    print("\nTasas de conversión:")
    print(conversion_pais)

    # Test chi-cuadrado para este país
    if tabla_pais.shape == (2, 2):  # Solo si tenemos ambas variantes
        chi2, p_val, _, _ = chi2_contingency(tabla_pais)
        print(f"\nChi-cuadrado: {chi2:.4f}")
        print(f"P-value: {p_val:.4f}")

        if p_val < 0.05:
            print("✅ SIGNIFICATIVO")
        else:
            print("❌ NO SIGNIFICATIVO")





🌍 ANÁLISIS PARA ARGENTINA:
Tabla de contingencia:
convirtio       0    1
variante              
control      1388  268
tratamiento  1388  273

Tasas de conversión:
             total_usuarios  conversiones  tasa_conversion  \
variante                                                     
control                1656           268         0.161836   
tratamiento            1661           273         0.164359   

             tasa_conversion_pct  
variante                          
control                16.183575  
tratamiento            16.435882  

Chi-cuadrado: 0.0224
P-value: 0.8810
❌ NO SIGNIFICATIVO

🌍 ANÁLISIS PARA MEXICO:
Tabla de contingencia:
convirtio       0    1
variante              
control      1425  252
tratamiento  1436  292

Tasas de conversión:
             total_usuarios  conversiones  tasa_conversion  \
variante                                                     
control                1677           252         0.150268   
tratamiento            1728           292

In [ ]:
# Análisis de conversión por dispositivo
for dispositivo in e_checkout['dispositivo'].unique():
    print(f"\n🌍 ANÁLISIS PARA {dispositivo.upper()}:")
    print("=" * 40)

    # Filtrar datos por país
    data_dispositivo = e_checkout[e_checkout['dispositivo'] == dispositivo]

    # Tabla de contingencia para este país
    tabla_dispositivo = pd.crosstab(data_dispositivo['variante'], data_dispositivo['convirtio'])
    print("Tabla de contingencia:")
    print(tabla_dispositivo)

    # Tasas de conversión por variante en este país
    conversion_dispositivo = data_dispositivo.groupby('variante')['convirtio'].agg(['count', 'sum', 'mean'])
    conversion_dispositivo.columns = ['total_usuarios', 'conversiones', 'tasa_conversion']
    conversion_dispositivo['tasa_conversion_pct'] = conversion_pais['tasa_conversion'] * 100
    print("\nTasas de conversión:")
    print(conversion_dispositivo)

    # Test chi-cuadrado para este país
    if tabla_dispositivo.shape == (2, 2):  # Solo si tenemos ambas variantes
        chi2, p_val, _, _ = chi2_contingency(tabla_dispositivo)
        print(f"\nChi-cuadrado: {chi2:.4f}")
        print(f"P-value: {p_val:.4f}")

        if p_val < 0.05:
            print("✅ SIGNIFICATIVO")
        else:
            print("❌ NO SIGNIFICATIVO")



🌍 ANÁLISIS PARA MOBILE:
Tabla de contingencia:
convirtio       0    1
variante              
control      2139  318
tratamiento  2142  359

Tasas de conversión:
             total_usuarios  conversiones  tasa_conversion  \
variante                                                     
control                2457           318         0.129426   
tratamiento            2501           359         0.143543   

             tasa_conversion_pct  
variante                          
control                15.870098  
tratamiento            15.492102  

Chi-cuadrado: 1.9768
P-value: 0.1597
❌ NO SIGNIFICATIVO

🌍 ANÁLISIS PARA DESKTOP:
Tabla de contingencia:
convirtio       0    1
variante              
control      2047  461
tratamiento  2073  461

Tasas de conversión:
             total_usuarios  conversiones  tasa_conversion  \
variante                                                     
control                2508           461         0.183812   
tratamiento            2534           461  

La prueba chi-cuadrado no mostró diferencias estadísticamente significativas entre las variantes (p > 0.05). Las tasas de conversión entre control y tratamiento son similares. Esto significa que los cambios implementados no generaron un impacto medible en la conversión. La muestra fue lo suficientemente grande, 4965 para la variante `control` y 5035 para la variante `tratamiento`. Sin embargo las conversiones fueron sólo del 15.7% para la variante `control`y 16.3% para la variante`tratamiento`. La segmentación por país y por dispositivo no cambió el resultado. No hay ninguna prueba de que la variante de tratamiento haya modificado la conversión. No recomendamos la implementación de los cambios en el producto de la variante Tratamiento.

---

---

In [ ]:
# link de tableau
#https://public.tableau.com/views/Sprint12_ProyectoFinal/DashboardOverview?:language=es-ES&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link
